In [1]:
%cd ~/Documents/GitHub/MMMM

/Users/matthewziyusu/Documents/GitHub/MMMM


In [2]:
import sys

sys.path.append("./models")

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from eeg_encoder import TextHead, EEGEncoder

# import torch
# import torch.nn as nn
# import torch.nn.functional as F

# class TextHead(nn.Module):
#     def __init__(self, input_dim=840, output_dim=1024, hidden_dim=1024, num_layers=2):
#         super(TextHead, self).__init__()
#         self.hidden_layers = nn.ModuleList()
#         self.hidden_layers.append(nn.Linear(input_dim, hidden_dim))
#         for i in range(num_layers - 1):
#             self.hidden_layers.append(nn.Linear(hidden_dim, hidden_dim))
#         self.output_layer = nn.Linear(hidden_dim, output_dim)
#         self.activation = nn.ReLU()

#     def forward(self, x):
#         for layer in self.hidden_layers:
#             x = self.activation(layer(x))
#         x = self.activation(self.output_layer(x))
#         return x

# class EEGEncoder(nn.Module):
#     def __init__(self, txt_in_feat=840, img_in_feat=500, enc_feat=1024, dec_emb_sz=768, enc_nhead=8, enc_dim_ff=2048, num_enc_layers=8):
#         super(EEGEncoder, self).__init__()

#         self.txt_head = TextHead(input_dim=txt_in_feat, output_dim=enc_feat)
#         self.img_head = nn.Linear(img_in_feat, enc_feat)

#         self.encoder_layer = nn.TransformerEncoderLayer(d_model=enc_feat, nhead=enc_nhead, dim_feedforward=enc_dim_ff, batch_first=True)
#         self.encoder = nn.TransformerEncoder(self.encoder_layer, num_layers=num_enc_layers)
#         self.fc_proj = nn.Linear(enc_feat, dec_emb_sz)

#     def forward(self, mode, input_data_batch=None, input_masks_batch=None, input_masks_invert=None):
#         if mode == "TXT":
#             encoded_embedding = self.txt_head(input_data_batch)
#         elif mode == "IMG":
#             encoded_embedding = self.img_head(input_data_batch)

#         encoded_embedding = self.encoder(input_data_batch, src_key_padding_mask = input_masks_invert)
#         encoded_embedding = F.relu(self.fc_proj(encoded_embedding))
#         return encoded_embedding

In [4]:
test_data = torch.randn(8, 10, 10)

In [5]:
text_hd = TextHead(10, 16)

In [6]:
with torch.no_grad():
    results = text_hd(test_data)
print(results)

tensor([[[0.0000, 0.0461, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.1388, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0752],
         [0.0662, 0.1277, 0.0000,  ..., 0.0000, 0.0000, 0.0595],
         ...,
         [0.0000, 0.1169, 0.0000,  ..., 0.0234, 0.0000, 0.2209],
         [0.0000, 0.0358, 0.0000,  ..., 0.0625, 0.0000, 0.0271],
         [0.0000, 0.0182, 0.0000,  ..., 0.1167, 0.0000, 0.0833]],

        [[0.0113, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0709],
         [0.0049, 0.0000, 0.0000,  ..., 0.0345, 0.0533, 0.0702],
         [0.0822, 0.0000, 0.0000,  ..., 0.0090, 0.0218, 0.0000],
         ...,
         [0.0000, 0.0126, 0.0000,  ..., 0.0000, 0.0000, 0.0671],
         [0.0176, 0.1285, 0.0000,  ..., 0.0000, 0.0000, 0.0372],
         [0.0000, 0.0847, 0.0000,  ..., 0.0000, 0.0000, 0.0763]],

        [[0.0000, 0.0081, 0.0000,  ..., 0.0628, 0.0221, 0.0949],
         [0.0835, 0.0000, 0.0000,  ..., 0.0016, 0.0000, 0.0003],
         [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.

In [7]:
eeg_enc = EEGEncoder(txt_in_feat=10, enc_feat=10, dec_emb_sz=16, enc_nhead = 1, enc_dim_ff=16, num_enc_layers=8)

/opt/homebrew/lib/python3.9/site-packages/torch/nn/modules/transformer.py:282: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


In [8]:
# masks = torch.ones(10)
# masks[0] = masks[1] = masks[2] = masks[3] = masks[4] = 0
masks = torch.zeros(8, 10)
inv_masks = torch.ones(8, 10)

for i in range(0, 8):
    masks[i][0] = masks[i][1] = masks[i][2] = masks[i][3] = masks[i][4] = 1
    inv_masks[i][0] = inv_masks[i][1] = inv_masks[i][2] = inv_masks[i][3] = inv_masks[i][4] = 0

In [9]:
results = eeg_enc(mode="TXT", input_embeddings_batch = test_data, input_masks_batch = masks, input_masks_invert = masks)

In [10]:
print(results)

tensor([[[0.3366, 0.0000, 0.0000,  ..., 0.5279, 1.2576, 0.0000],
         [0.0000, 0.0000, 0.0000,  ..., 1.0186, 0.5896, 0.0000],
         [0.0000, 0.0000, 0.0000,  ..., 0.8832, 1.0193, 0.0000],
         ...,
         [0.6479, 0.0000, 0.0000,  ..., 0.3968, 0.6734, 0.0000],
         [0.0000, 0.0000, 0.3244,  ..., 1.2696, 0.7725, 0.0000],
         [0.0000, 0.0832, 0.0000,  ..., 1.1444, 0.6983, 0.0000]],

        [[0.0000, 0.0532, 0.1569,  ..., 1.0765, 0.1903, 0.2819],
         [0.0000, 0.0000, 0.2508,  ..., 1.2930, 0.2300, 0.2778],
         [0.0000, 0.0295, 0.0000,  ..., 1.3894, 0.1241, 0.2257],
         ...,
         [0.0000, 0.0000, 0.0000,  ..., 0.0357, 0.2919, 0.0000],
         [0.1730, 0.0000, 0.0000,  ..., 0.1895, 0.4154, 0.0000],
         [0.0000, 0.0000, 0.0000,  ..., 0.3720, 0.3632, 0.2187]],

        [[0.0000, 0.3886, 0.2233,  ..., 1.3009, 0.0000, 0.2493],
         [0.7669, 0.0000, 0.0000,  ..., 0.1081, 0.5784, 0.0000],
         [0.5610, 0.0000, 0.0000,  ..., 0.2500, 1.0711, 0.

In [11]:
results.shape

torch.Size([8, 10, 16])